## Snakemake for Converting GPEP Forcing to Snakemake ##

Using [snakemake](https://snakemake.readthedocs.io/en/stable/) to wrap a meteorological data processing workflow

### Main snakemake file - gpep_to_summa

The Snakemake file below can be used to run the entire data processing workflow.
The Snakemake file is first written, then run from the command line.

#### Write the snakemake code below to file

Note that this is python code, which some additional syntax for snakemake.

In [4]:
config_file = '../config/gpep_to_summa_hydrofabric_config.yaml'

In [1]:
%%writefile ../gpep_to_summa_multiple.smk
''' 
gmet to summa snakemake master snakemake file

This snakemake file runs all the steps required to convert GPEP forcings to SUMMA forcings.

Original process code: Andy Wood
Adapted to Snakemake: Dave Casson

'''
from pathlib import Path
from scripts import gpep_to_summa_utils as utils

configfile: '../config/gpep_to_summa_bow_above_banff.yaml'

case_name = config['case_name']
result_dir = config['result_dir']

print(config)
config['gpep_forcing_dir'] =  Path(result_dir, case_name, 'ensembles')
config['gpep_tmp_forcing_dir'] =  Path(result_dir, case_name,'tmp')
config['working_dir'] = Path(result_dir, case_name, 'summa_forcing')
config['summa_dir'] = Path(result_dir, case_name, 'summa_forcing','summa')
config['summa_forcing_dir'] = Path(result_dir, case_name, 'summa_forcing','summa')


# Resolve paths from the configuration file
config = utils.resolve_paths(config, log_config = True,resolve_config = True)

# Read all forcing files and create a list based on the output directory (i.e. ens/filename.nc)
_, file_path_list = utils.build_ensemble_list(config['gpep_forcing_dir'])
file_path_list = list(file_path_list)

# Read in all local snakemake files and rules
include: './rules/gpep_file_prep.smk'
include: './rules/remap_gpep_to_shp.smk'
include: './rules/metsim_file_prep.smk'
include: './rules/run_metsim.smk'
include: './rules/metsim_to_summa.smk'

ruleorder: 
    gpep_to_summa >
    gpep_file_prep >
    remap_gpep_to_shp >
    metsim_file_prep >
    run_metsim >
    metsim_to_summa
    
# Run the snakemake file, so that that it produces a summa input file for each of the gpep forcing files
rule gpep_to_summa:
    input:
        expand(Path(result_dir, case_name, 'summa_forcing','summa','{forcing_file}.nc'), forcing_file = file_path_list)

Overwriting ../gpep_to_summa_multiple.smk


### Perform a Dry Run, and unlock the working directory

In [5]:
## Perform a dry run to check the workflow is configured correctly.
! snakemake --unlock -s ../gpep_to_summa_multiple.smk --configfile {config_file}
! snakemake -s ../gpep_to_summa_multiple.smk --configfile {config_file} --dry-run

Settings logged from {'case_name': 'lwr_static_boxcox_low_predictors', 'gpep_forcing_dir': PosixPath('/scratch/dcasson/gpep/transform_tests/lwr_static_boxcox_low_predictors/ensembles'), 'working_dir': PosixPath('/scratch/dcasson/gpep/transform_tests/lwr_static_boxcox_low_predictors/summa_forcing'), 'summa_dir': PosixPath('/scratch/dcasson/gpep/transform_tests/lwr_static_boxcox_low_predictors/summa_forcing/summa'), 'summa_forcing_dir': PosixPath('/scratch/dcasson/gpep/transform_tests/lwr_static_boxcox_low_predictors/summa_forcing/summa'), 'input_files': None, 'catchment_shp': '/home/dcasson/gpep_to_summa/gpkg/tuolumne_hydrofabric_gru.gpkg', 'attribute_nc': '/home/dcasson/gpep_to_summa/gpkg/tuolumne_hydrofabric_gru_attributes.nc', 'metsim_base_config': '../config/metsim_base_config.yaml', 'catchment_shp_hru_id_field': 'HRU_ID', 'catchment_shp_lat_id_field': 'latitude', 'catchment_shp_lon_id_field': 'longitude', 'forcing_shp': 'forcing_shp.shp', 'easymore_input_var': ['prcp', 't_max', 't_

## Run the complete gmet to summa snakemake workflow

In [ ]:
! snakemake -s ../gpep_to_summa_multiple.smk -c 8 --configfile {config_file}

### Visualize the workflow rules

Note that the graphviz must be installed

In [ ]:
from IPython import display
# Buold build the rule graph
! snakemake -s ../gpep_to_summa.smk --configfile {config_file} --rulegraph | dot -Tpng > ../reports/gpep_to_summa_rule.png
# Python command to visualise the built image in our notebook
display.Image('../reports/gpep_to_summa_rule.png')


### Visualize the workflow files

Note that the graphviz must be installed

In [ ]:
# Build the file graph
! snakemake -s ../gpep_to_summa.smk --configfile {config_file} --filegraph | dot -Tpng > ../reports/gpep_to_summa_file.png
display.Image('../reports/gpep_to_summa_file.png')